In [1]:
import json
import os

In [22]:
# mut_rels = ['P6', 'P39', 'P54', 'P108', 'P210', 'P264', 'P286', 'P451', 'P488', 'P551', 'P937', 'P1037', 'P1308']
# imm_1_rels = ['P19', 'P20', 'P30', 'P36', 'P103', 'P138', 'P140', 'P159', 'P364', 'P449', 'P495', 'P740']
# imm_n_rels = ['P27', 'P47', 'P69', 'P101', 'P136', 'P166', 'P190', 'P530', 'P1303', 'P1412']

mut_rels = ['P106T', 'P19T', 'P136T', 'P22T', 'P17T', 'P162T', 'P57T', 'P1376T', 'P58T', 'P86T', 'P462T', 'P140T', 'P641T', 'P50T', 'P25T', 'P36T']
imm_1_rels = ['P106S', 'P19S', 'P136S', 'P22S', 'P17S', 'P162S', 'P57S', 'P1376S', 'P58S', 'P86S', 'P462S', 'P140S', 'P641S', 'P50S', 'P25S', 'P36S']
imm_n_rels = []

In [23]:
with open("dqa_rel_dict.json", "r") as file:
    # Load JSON data into a dictionary
    data = json.load(file)

# Now 'data' is a Python dictionary
print(data)

{'immutable': ['P106S', 'P19S', 'P136S', 'P22S', 'P17S', 'P162S', 'P57S', 'P1376S', 'P58S', 'P86S', 'P462S', 'P140S', 'P641S', 'P50S', 'P25S', 'P36S'], 'mutable': ['P106T', 'P19T', 'P136T', 'P22T', 'P17T', 'P162T', 'P57T', 'P1376T', 'P58T', 'P86T', 'P462T', 'P140T', 'P641T', 'P50T', 'P25T', 'P36T']}


In [29]:
model_name = 'llamachat_dqa'

In [30]:
# print(len(mut_rels)+len(imm_1_rels)+len(imm_n_rels))

In [31]:
mut_count = imm_1_count = imm_n_count = 0
mut_sum = imm_1_sum = imm_n_sum = 0
for template_num in range(5):
    template_dir = f'{model_name}_{template_num}'
    # with open(f'{template_dir}/all_results_per_example.json', 'r') as file:
    #     data = json.load(file)
    #     print(data.keys())
    #     print(len(data['id']))
    #     print(len(data['prediction']))
    #     print(len(data['ground_truth']))
    #     print(len(data['f1']))
    #     print(len(data['exact_match']))
    fname = f'{template_dir}/metrics.jsonl'
    with open(fname, 'r') as file:
        print(fname)
        data = [json.loads(line) for line in file]
        for record in data:
            if 'all' in record:
                print(record)

llamachat_dqa_0/metrics.jsonl
{'all': 20.801391559836432}
llamachat_dqa_1/metrics.jsonl
{'all': 24.967879652392078}
llamachat_dqa_2/metrics.jsonl
{'all': 22.925581959998787}
llamachat_dqa_3/metrics.jsonl
{'all': 24.084865161984343}
llamachat_dqa_4/metrics.jsonl
{'all': 23.72543916202297}


In [32]:
mut_count = imm_1_count = imm_n_count = 0
mut_sum = imm_1_sum = imm_n_sum = 0
for template_num in range(5):
    # list relation files in a template dir
    template_dir = f'{model_name}_{template_num}'
    files = [f for f in os.listdir(template_dir) 
         if os.path.isfile(os.path.join(template_dir, f)) and f.startswith('P')]
    print(template_dir)

    # merge metrics dicts from metrics.jsonl
    metrics_fname = f'{template_dir}/metrics.jsonl'
    metrics_dict = {}
    with open(metrics_fname, 'r') as file:
        print(metrics_fname)
        metrics_records = [json.loads(line) for line in file]
        for d in  metrics_records:
            for k, v in d.items():
                if k in metrics_dict:
                    raise ValueError(f"Duplicate key found: {k}")
                metrics_dict[k] = v

    # loop over mutable relations
    for rel in mut_rels:
        rel_template_f1 = metrics_dict[rel]
        # print(f'{rel}: {rel_template_f1}')
        rel_fname = f'{template_dir}/{rel}_results_per_example.json'

        with open(rel_fname, 'r') as file:
            data = json.load(file)
            # print(data.keys())
            # print(len(data['id']))
            sample_count = len(data['id'])
            # print(sample_count)
            mut_count += sample_count
            mut_sum += rel_template_f1 * sample_count

    # loop over immutable-1 relations
    for rel in imm_1_rels:
        rel_template_f1 = metrics_dict[rel]
        # print(f'{rel}: {rel_template_f1}')
        rel_fname = f'{template_dir}/{rel}_results_per_example.json'

        with open(rel_fname, 'r') as file:
            data = json.load(file)
            # print(data.keys())
            # print(len(data['id']))
            sample_count = len(data['id'])
            # print(sample_count)
            imm_1_count += sample_count
            imm_1_sum += rel_template_f1 * sample_count

    # loop over immutable-N relations
    for rel in imm_n_rels:
        rel_template_f1 = metrics_dict[rel]
        # print(f'{rel}: {rel_template_f1}')
        rel_fname = f'{template_dir}/{rel}_results_per_example.json'

        with open(rel_fname, 'r') as file:
            data = json.load(file)
            # print(data.keys())
            # print(len(data['id']))
            sample_count = len(data['id'])
            # print(sample_count)
            imm_n_count += sample_count
            imm_n_sum += rel_template_f1 * sample_count

mut_avg_f1 = mut_sum / mut_count
print(mut_avg_f1)
imm_1_avg_f1 = imm_1_sum / imm_1_count
print(imm_1_avg_f1)
# imm_n_avg_f1 = imm_n_sum / imm_n_count
# print(imm_n_avg_f1)

llamachat_dqa_0
llamachat_dqa_0/metrics.jsonl
llamachat_dqa_1
llamachat_dqa_1/metrics.jsonl
llamachat_dqa_2
llamachat_dqa_2/metrics.jsonl
llamachat_dqa_3
llamachat_dqa_3/metrics.jsonl
llamachat_dqa_4
llamachat_dqa_4/metrics.jsonl
24.16121863335275
22.541658297258294


In [33]:
print(f"{imm_1_avg_f1=}")
# print(f"{imm_n_avg_f1=}")
print(f"{mut_avg_f1=}")

imm_1_avg_f1=22.541658297258294
mut_avg_f1=24.16121863335275
